In [0]:
%sql
create catalog if not exists sales;

In [0]:
%sql
use catalog sales;

create schema if not exists bronze;
create schema if not exists silver;
create schema if not exists gold;


In [0]:
%sql
use catalog sales;
use schema bronze;

create table if not exists sales_transactions(
    transaction_id integer,
    customer_id string,
    product string,
    category string,
    amount double,
    transaction_date date,
    status string
);

In [0]:
%sql
insert into sales.bronze.sales_transactions values 
    (1001, 'C001', 'laptop', 'electronics', 1200.00, '2026-01-05', 'completed'),
    (1002, 'C002', 'smartphone', 'electronics', 800.00, '2026-01-07', 'completed'),
    (1003, 'C001', 'mouse', 'accessories', 50.00, '2026-01-07', 'cancelled'),
    (1004, 'C003', 'desk', 'furniture', 300.00, '2026-01-08', 'completed'),
    (1005, 'C002', 'keyboard', 'furniture', 200.00, '2026-01-09', 'completed')

In [0]:
%sql
describe table sales.bronze.sales_transactions

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [0]:
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()


In [0]:
df = spark.sql ("select * from sales.bronze.sales_transactions")

In [0]:
cleaned_df = df.dropDuplicates(["transaction_id"])

In [0]:
display(cleaned_df)

In [0]:
cleaned_df.write.format("delta").mode("overwrite").saveAsTable("sales.silver.sales_transactions")

In [0]:
%sql
select * from sales.silver.sales_transactions;

In [0]:
sales_df = spark.table("sales.silver.sales_transactions")

In [0]:
cust_sales = (
    sales_df
    .filter(col("status") == "completed")
    .groupBy("customer_id")
    .agg(sum("amount").alias("total_sales"))
)

#### Using Window functions

In [0]:
%sql 
select *
from (
    select *,
        row_number() over(partition by customer_id order by amount desc) as rn
        from sales.silver.sales_transactions
        where status = 'completed'
)
where rn = 1;



In [0]:
df = spark.sql("""
    select transaction_id,customer_id,product,category,amount,transaction_date
from (
    select *,
        row_number() over(partition by customer_id order by amount desc) as rn
        from sales.silver.sales_transactions
        where status = 'completed'
)    
where rn = 1              
""")

In [0]:
df.show()

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("sales.gold.top_purchases")

In [0]:
%sql
select * from sales.gold.top_purchases

In [0]:
%sql
create table if not exists sales.bronze.customers (
    customer_id string,
    customer_name string
)

In [0]:
%sql
insert into sales.bronze.customers values 
('C0001', 'Alice'),
('C002', 'Bob'),
('C003', 'John')

In [0]:
%sql

update sales.bronze.customers
set customer_id = 'C001'
where customer_name = 'Alice'

In [0]:
%sql
use catalog sales;
use schema bronze;
    
    
select st.*,c.customer_name from customers c
left join sales_transactions st on c.customer_id = st.customer_id;

In [0]:
joineddf = spark.sql("""
    select st.*,c.customer_name from sales.bronze.customers c
    left join sales.bronze.sales_transactions st on c.customer_id = st.customer_id;
    """)

In [0]:
joineddf.write.format("delta").mode("overwrite").saveAsTable("sales.silver.final_df")

In [0]:
%sql
use catalog sales;
use schema silver;

show tables;

In [0]:
%sql
use catalog sales;
use schema gold;

select * from top_purchases;